In [17]:
import pandas as pd
from collections import Counter

# Load the datasets
df1 = pd.read_csv("datasets/Raw_Datasets/zeekdata22fall.csv")
df2 = pd.read_csv("datasets/Raw_Datasets/zeekdata24.csv")
df3 = pd.read_csv("datasets/Raw_Datasets/zeekdata24fall.csv")
df4 = pd.read_csv("datasets/Updated/final_train.csv") #Reconnaissance combined - train set
df5 = pd.read_csv("datasets/Updated/test.csv") #Reconnaissance combined - test set
df6 = pd.read_csv("datasets/final_train.csv") #Resource Development combined - train set
df7 = pd.read_csv("datasets/test.csv") #Resource Development combined - test set

datasets = {
    "zeekdata22fall": df1,
    "zeekdata24": df2,
    "zeekdata24fall": df3
}

C:\Users\Admin\AppData\Local\Temp\ipykernel_17348\2274206370.py:5: DtypeWarning: Columns (0: service) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv("datasets/Raw_Datasets/zeekdata22fall.csv")


In [18]:
def combined_unique_values(column_name, *dfs):
    values = set()

    for df in dfs:
        if column_name in df.columns:
            values.update(df[column_name].dropna().unique())

    values = sorted(values)

    print(f"\nCombined unique values in '{column_name}' ({len(values)}):")
    for value in values:
        print(value)

    return values


In [19]:
def combined_unique_values_with_counts(column_name, *dfs):
    counter = Counter()

    
    for df in dfs:
        if column_name in df.columns:
            values = df[column_name].dropna().values
            counter.update(values)

    
    sorted_items = counter.most_common()

    print(f"\nCombined value counts for '{column_name}':")
    for value, count in sorted_items:
        print(f"{value}: {count}")

    
    return dict(counter)

In [20]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(name)
    print(f"{'='*50}")

    # Missing values per column
    missing_count = df.isnull().sum()

    # Keep only columns that have missing values
    missing_count = missing_count[missing_count > 0]

    if missing_count.empty:
        print("No missing values found.")
        continue

    # Calculate percentages only for columns with missing values
    missing_percent = (missing_count / len(df)) * 100

    summary = pd.DataFrame({
        "Missing Count": missing_count,
        "Missing Percentage (%)": missing_percent.round(2)
    })

    print("\nColumns with missing values:")
    print(summary)

    # Number of rows with at least one missing value
    rows_with_missing = df.isnull().any(axis=1).sum()
    print(f"\nInstances with at least one missing value: {rows_with_missing}")
    print(f"Total instances: {len(df)}")


zeekdata22fall

Columns with missing values:
            Missing Count  Missing Percentage (%)
duration            15705                   45.05
history               415                    1.19
orig_bytes          15705                   45.05
resp_bytes          15705                   45.05
service             15836                   45.43

Instances with at least one missing value: 16078
Total instances: 34860

zeekdata24

Columns with missing values:
            Missing Count  Missing Percentage (%)
duration             2871                    2.99
history                61                    0.06
orig_bytes           2871                    2.99
resp_bytes           2871                    2.99
service              7721                    8.05

Instances with at least one missing value: 9842
Total instances: 95871

zeekdata24fall

Columns with missing values:
            Missing Count  Missing Percentage (%)
duration            62797                   43.11
history              

In [21]:
def analyze_label_characteristics(label_column, label_value, *dfs, min_frequency=0.90):
    # Combine all matching rows
    combined = pd.concat(
        [df[df[label_column] == label_value] for df in dfs],
        ignore_index=True
    )

    if combined.empty:
        print(f"No samples found for '{label_value}'.")
        return

    print("=" * 100)
    print(f"Analysis for '{label_value}'")
    print(f"Total Samples: {len(combined)}")
    print("=" * 100)

    results = []

    for col in combined.columns:

        if col == label_column:
            continue

        counts = combined[col].value_counts(dropna=False)

        if counts.empty:
            continue

        most_common_value = counts.index[0]
        most_common_count = counts.iloc[0]
        percentage = (most_common_count / len(combined)) * 100

        # Skip columns where every value is unique
        if most_common_count == 1:
            continue

        if percentage >= min_frequency * 100:
            results.append({
                "Column": col,
                "Most Common Value": most_common_value,
                "Count": most_common_count,
                "Percentage": percentage
            })

    if not results:
        print(f"No columns met the {min_frequency*100:.0f}% threshold.")
        return

    result_df = (
        pd.DataFrame(results)
        .sort_values(by="Percentage", ascending=False)
        .reset_index(drop=True)
    )

    print(result_df.to_string(index=False))

In [22]:
protocol_unique_values = combined_unique_values("proto", df1, df2, df3)
label_tactics = combined_unique_values("label_tactic", df1, df2, df3)
attack_counts = combined_unique_values_with_counts("label_tactic", df1, df2, df3)




Combined unique values in 'proto' (3):
icmp
tcp
udp

Combined unique values in 'label_tactic' (15):
Collection
Command and Control
Credential Access
Defense Evasion
Discovery
Execution
Exfiltration
Initial Access
Lateral Movement
Persistence
Priviledge Escalation
Privilege Escalation
Reconnaissance
Resource Development
none

Combined value counts for 'label_tactic':
none: 138167
Discovery: 67585
Credential Access: 43495
Resource Development: 13644
Reconnaissance: 10690
Defense Evasion: 873
Initial Access: 617
Privilege Escalation: 547
Persistence: 336
Priviledge Escalation: 326
Lateral Movement: 40
Execution: 32
Exfiltration: 23
Command and Control: 20
Collection: 1


In [23]:
column_name = "label_tactic"
for name, df in datasets.items():
    print(f"\n{name}")
    print(df[column_name].value_counts(dropna=False))


zeekdata22fall
label_tactic
none                    17508
Resource Development    13644
Reconnaissance           2492
Discovery                 861
Defense Evasion           133
Privilege Escalation      133
Execution                  30
Initial Access             19
Command and Control        17
Lateral Movement           11
Persistence                10
Collection                  1
Credential Access           1
Name: count, dtype: int64

zeekdata24
label_tactic
none                     47906
Credential Access        43494
Reconnaissance            2909
Initial Access             561
Persistence                326
Priviledge Escalation      326
Defense Evasion            326
Exfiltration                23
Name: count, dtype: int64

zeekdata24fall
label_tactic
none                    72753
Discovery               66724
Reconnaissance           5289
Defense Evasion           414
Privilege Escalation      414
Initial Access             37
Lateral Movement           29
Command and Contr

In [24]:
print ("Reconnaissance combined - test set")
for tactic in sorted(set(df5["label_tactic"])):
    analyze_label_characteristics("label_tactic",tactic,df5)
    print("\n")

Reconnaissance combined - test set
Analysis for 'Credential Access'
Total Samples: 8699
                Column  Most Common Value  Count  Percentage
            local_resp                  0   8699  100.000000
        src_port_bin_1                  0   8699  100.000000
        src_port_bin_2                  1   8699  100.000000
        src_port_bin_3                  0   8699  100.000000
       dest_port_bin_1                  0   8699  100.000000
       dest_port_bin_2                  1   8699  100.000000
       dest_port_bin_3                  0   8699  100.000000
      duration_bin_0.0                  0   8699  100.000000
    orig_bytes_bin_1.0                  1   8699  100.000000
    orig_bytes_bin_2.0                  0   8699  100.000000
  missed_bytes_bin_4.0                  0   8699  100.000000
    orig_bytes_bin_6.0                  0   8699  100.000000
     orig_pkts_bin_1.0                  0   8699  100.000000
    resp_bytes_bin_0.0                  0   8699  100.0000

In [25]:
print ("Resource Development combined - test set")
for tactic in sorted(set(df7["label_tactic"])):
    analyze_label_characteristics("label_tactic",tactic,df7)
    print("\n")

Resource Development combined - test set
Analysis for 'Credential Access'
Total Samples: 8699
               Column  Most Common Value  Count  Percentage
           local_resp                  0   8699  100.000000
       src_port_bin_1                  0   8699  100.000000
       src_port_bin_2                  1   8699  100.000000
       src_port_bin_3                  0   8699  100.000000
      dest_port_bin_1                  0   8699  100.000000
      dest_port_bin_2                  1   8699  100.000000
      dest_port_bin_3                  0   8699  100.000000
     duration_bin_0.0                  0   8699  100.000000
 missed_bytes_bin_3.0                  0   8699  100.000000
     conn_state_RSTRH                  0   8699  100.000000
   orig_bytes_bin_0.0                  0   8699  100.000000
   orig_bytes_bin_1.0                  1   8699  100.000000
    orig_pkts_bin_1.0                  0   8699  100.000000
   resp_bytes_bin_0.0                  0   8699  100.000000
      